[Reference](https://blog.devgenius.io/10-pyspark-data-validation-patterns-used-by-data-analysts-before-writing-to-lakehouse-059ad059ccef$0)

# 1. Validate Schema Evolution Before Write


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

# Define expected schema with strict nullability and metadata
expected_schema = StructType([
    StructField("transaction_id", StringType(), False, {"description": "Unique transaction identifier"}),
    StructField("amount", DoubleType(), False, {"precision": "2 decimals"}),
    StructField("event_timestamp", TimestampType(), False, {}),
    StructField("customer_id", StringType(), True, {})
])

# Deep schema comparison function
def validate_schema_strict(df, expected):
    actual_schema = df.schema

    # Check field count
    if len(actual_schema) != len(expected):
        raise ValueError(f"Field count mismatch: expected {len(expected)}, got {len(actual_schema)}")

    # Check each field name, type, and nullability
    for actual_field, expected_field in zip(actual_schema, expected):
        if actual_field.name != expected_field.name:
            raise ValueError(f"Field name mismatch: expected '{expected_field.name}', got '{actual_field.name}'")
        if actual_field.dataType != expected_field.dataType:
            raise ValueError(f"Type mismatch for '{actual_field.name}': expected {expected_field.dataType}, got {actual_field.dataType}")
        if actual_field.nullable != expected_field.nullable:
            raise ValueError(f"Nullability mismatch for '{actual_field.name}': expected {expected_field.nullable}, got {actual_field.nullable}")

    return True

# Usage with error collection and reporting
try:
    validate_schema_strict(df, expected_schema)
    print("Schema validation passed")
except ValueError as e:
    # Log detailed error and fail pipeline
    print(f"Schema drift detected: {str(e)}")
    # Trigger alerting mechanism
    raise

# For evolving schemas, check compatibility instead of exact match
def validate_schema_compatible(df, expected):
    actual_fields = {f.name: f.dataType for f in df.schema}
    expected_fields = {f.name: f.dataType for f in expected}

    # Ensure all expected fields exist with correct types
    for name, dataType in expected_fields.items():
        if name not in actual_fields:
            raise ValueError(f"Missing required field: {name}")
        if actual_fields[name] != dataType:
            raise ValueError(f"Type mismatch for {name}: expected {dataType}, got {actual_fields[name]}")

    return True

# 2. Validate Statistical Distributions (Not Just Min/Max)


In [3]:
from pyspark.sql.functions import skewness, kurtosis, stddev, mean, abs as Fabs

def validate_distribution_metrics(df, column, historical_stats=None, tolerance=0.15):
    """
    Validate statistical distribution against thresholds or historical baseline
    historical_stats: dict with mean, stddev, skewness from previous runs
    """
    # Calculate comprehensive statistics
    stats = df.select(
        mean(col(column)).alias("mean"),
        stddev(col(column)).alias("stddev"),
        skewness(col(column)).alias("skewness"),
        kurtosis(col(column)).alias("kurtosis"),
        min(col(column)).alias("min"),
        max(col(column)).alias("max")
    ).collect()[0]

    # Calculate quantiles (0.01, 0.05, 0.5, 0.95, 0.99)
    quantiles = df.approxQuantile(column, [0.01, 0.05, 0.5, 0.95, 0.99], 0.01)

    # Detect outliers using IQR method
    q1, q3 = quantiles[1], quantiles[3]
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = df.filter(
        (col(column) < lower_bound) | (col(column) > upper_bound)
    ).count()

    # Check against historical baseline if provided
    drift_detected = False
    if historical_stats:
        mean_drift = abs(stats["mean"] - historical_stats["mean"]) / historical_stats["mean"]
        std_drift = abs(stats["stddev"] - historical_stats["stddev"]) / historical_stats["stddev"]

        if mean_drift > tolerance or std_drift > tolerance:
            drift_detected = True

    return {
        "column": column,
        "mean": stats["mean"],
        "stddev": stats["stddev"],
        "skewness": stats["skewness"],
        "kurtosis": stats["kurtosis"],
        "quantiles": dict(zip(["p01", "p05", "p50", "p95", "p99"], quantiles)),
        "outlier_count": outlier_count,
        "outlier_percentage": (outlier_count / df.count()) * 100,
        "drift_detected": drift_detected,
        "bounds": {"lower": lower_bound, "upper": upper_bound}
    }

# Usage for multiple columns with anomaly detection
columns_to_check = ["order_amount", "discount_pct", "shipping_days"]
historical_baselines = {
    "order_amount": {"mean": 150.0, "stddev": 45.0},
    "discount_pct": {"mean": 5.0, "stddev": 2.5}
}

validation_results = []
for col_name in columns_to_check:
    hist_stats = historical_baselines.get(col_name)
    result = validate_distribution_metrics(df, col_name, hist_stats)
    validation_results.append(result)

# Fail if any critical column has >5% outliers
critical_violations = [
    r for r in validation_results
    if r["outlier_percentage"] > 5.0
]

if critical_violations:
    raise ValueError(f"Distribution anomalies detected in {len(critical_violations)} columns")

# 3. Enforce Null-Value Thresholds


In [4]:
from pyspark.sql.functions import col, sum, when, lit
from pyspark.sql.types import IntegerType, DoubleType

def validate_null_thresholds(df, thresholds, abort_on_failure=True):
    """
    Validate null percentages against configurable thresholds
    thresholds: dict of {column_name: max_null_percentage}
    """
    total_records = df.count()

    # Calculate null percentages for specified columns
    null_counts = df.select([
        (sum(col(c).isNull().cast("int")) / lit(total_records) * 100).alias(f"{c}_null_pct")
        for c in thresholds.keys()
    ]).collect()[0]

    violations = []
    for col_name, max_threshold in thresholds.items():
        actual_null_pct = null_counts[f"{col_name}_null_pct"]
        if actual_null_pct > max_threshold:
            violations.append({
                "column": col_name,
                "null_percentage": round(actual_null_pct, 2),
                "threshold": max_threshold,
                "status": "VIOLATED"
            })

    # Generate validation report
    validation_df = spark.createDataFrame(violations) if violations else spark.createDataFrame([],
        "column:string,null_percentage:double,threshold:double,status:string")

    # Decision logic
    if violations and abort_on_failure:
        raise ValueError(f"Null threshold violations found: {len(violations)} columns exceed limits")

    return validation_df

# Usage with different thresholds per column
threshold_config = {
    "customer_id": 0.0,      # Zero tolerance for key fields
    "email": 2.0,            # Allow minimal missing emails
    "phone": 5.0,            # Allow some missing phone numbers
    "optional_field": 50.0   # High tolerance for optional data
}

null_report = validate_null_thresholds(df, threshold_config, abort_on_failure=False)
null_report.show()

# For incremental pipelines, compare with historical patterns
def detect_null_anomaly(df, column, historical_avg_null_pct, std_dev_threshold=2):
    """Detect if current null rate is anomalous compared to historical average"""
    current_null_pct = df.filter(col(column).isNull()).count() / df.count() * 100
    deviation = abs(current_null_pct - historical_avg_null_pct)

    if deviation > std_dev_threshold * 5:  # Assuming 5% std dev
        return True, current_null_pct
    return False, current_null_pct

# 4. Detect Phantom Duplicates with Composite Keys


In [5]:
from pyspark.sql.functions import sha2, concat_ws, count, lit, when, col

def detect_composite_duplicates(df, key_columns, fuzzy_match=False, similarity_threshold=0.9):
    """
    Detect duplicates using composite key hashing
    fuzzy_match: enables string similarity matching for text fields
    """
    # Create composite hash
    df_with_hash = df.withColumn(
        "composite_hash",
        sha2(concat_ws("|", *[col(c).cast("string") for c in key_columns]), 256)
    )

    # Count duplicates
    duplicate_stats = df_with_hash.groupBy("composite_hash").agg(
        count(lit(1)).alias("duplicate_count")
    )

    # Flag records with duplicates
    df_flagged = df_with_hash.join(
        duplicate_stats.filter(col("duplicate_count") > 1),
        on="composite_hash",
        how="left"
    ).withColumn("is_duplicate", when(col("duplicate_count") > 1, True).otherwise(False))

    # For near-duplicates with fuzzy matching on string columns
    if fuzzy_match:
        from pyspark.sql.functions import levenshtein

        # Self-join on similar keys (expensive operation)
        string_keys = [c for c in key_columns if "name" in c or "address" in c]

        if string_keys:
            # Create pairs for similarity comparison (use with caution on large datasets)
            df_self = df_flagged.select(
                [col(c).alias(f"{c}_1") for c in key_columns] +
                [col("composite_hash").alias("hash_1")]
            )

            # Filter to potential matches (same zip, similar names)
            near_duplicates = df_self.join(
                df_flagged.withColumnRenamed("composite_hash", "hash_2"),
                (df_self.zip_code_1 == df_flagged.zip_code)
            ).filter(
                levenshtein(col("customer_name_1"), col("customer_name")) < 3
            ).select("hash_1", "hash_2")

    duplicate_summary = df_flagged.groupBy("is_duplicate").count()
    return df_flagged, duplicate_summary

# Usage with composite business keys
key_cols = ["customer_id", "last_name", "zip_code"]
df_with_dupes, dupe_summary = detect_composite_duplicates(df, key_cols)

# For incremental loads, check against existing golden records
def check_against_golden_records(df_new, df_golden, key_columns):
    """Ensure new records do not duplicate existing golden records"""
    new_hashes = df_new.withColumn("hash", sha2(concat_ws("|", *key_columns), 256))
    golden_hashes = df_golden.withColumn("hash", sha2(concat_ws("|", *key_columns), 256))

    existing_duplicates = new_hashes.join(golden_hashes.select("hash"), on="hash", how="inner")

    return existing_duplicates.count() == 0, existing_duplicates

# 5. Timezone-Check Timestamps (Before They Mix)


In [6]:
from pyspark.sql.functions import to_utc_timestamp, from_utc_timestamp, lit, coalesce
from pyspark.sql.types import TimestampType

def validate_and_normalize_timestamps(df, timestamp_columns, source_tz_column=None, default_tz="UTC"):
    """
    Validate timezone information and normalize all timestamps to UTC
    source_tz_column: column containing source timezone (e.g., 'America/New_York')
    """
    validation_report = []

    for ts_col in timestamp_columns:
        # Check if timestamps already contain timezone info
        has_tz = df.filter(
            col(ts_col).cast("string").rlike(r"[+\-]\d{2}:\d{2}$") |
            col(ts_col).cast("string").contains("GMT") |
            col(ts_col).cast("string").contains("UTC")
        ).count()

        total_count = df.count()
        tz_percentage = (has_tz / total_count) * 100 if total_count > 0 else 0

        # Normalize to UTC based on available timezone info
        if source_tz_column:
            # Convert from source timezone to UTC
            df = df.withColumn(
                f"{ts_col}_utc",
                to_utc_timestamp(col(ts_col), coalesce(col(source_tz_column), lit(default_tz)))
            )
        else:
            # Assume timestamps are in default timezone and convert to UTC
            df = df.withColumn(
                f"{ts_col}_utc",
                to_utc_timestamp(col(ts_col), default_tz)
            )

        # Validate no future timestamps (business rule example)
        from pyspark.sql.functions import current_timestamp
        future_count = df.filter(col(f"{ts_col}_utc") > current_timestamp()).count()

        validation_report.append({
            "column": ts_col,
            "has_timezone_info_pct": round(tz_percentage, 2),
            "future_timestamps": future_count,
            "status": "VALID" if tz_percentage == 100 and future_count == 0 else "WARNING"
        })

    return df, validation_report

# Usage with mixed timezone data
timestamp_cols = ["event_time", "transaction_time", "user_signup_time"]
df_normalized, tz_report = validate_and_normalize_timestamps(
    df,
    timestamp_cols,
    source_tz_column="region",  # Column contains timezone names
    default_tz="UTC"
)

# For timezone validation across multiple regions
def validate_timezone_consistency(df, timestamp_col, region_col):
    """Ensure all timestamps in same region have consistent timezone offsets"""
    tz_by_region = df.groupBy(region_col).agg(
        count(lit(1)).alias("record_count"),
        (sum(when(col(timestamp_col).cast("string").contains("UTC"), 1).otherwise(0)) / count(lit(1)) * 100).alias("utc_percentage")
    )

    # Flag regions with inconsistent timezone data
    inconsistent_regions = tz_by_region.filter(
        (col("utc_percentage") > 0) & (col("utc_percentage") < 100)
    )

    return tz_by_region, inconsistent_regions

# 6. Validate Enum/Categorical Values Against a Golden Set


In [7]:
from pyspark.sql.functions import broadcast, collect_set, size, array_except, lit
from pyspark.sql.types import ArrayType, StringType

def validate_categorical_values(df, category_rules, allow_unknowns=False):
    """
    Validate categorical columns against golden sets with support for hierarchical rules
    category_rules: dict of {column_name: {"valid_values": list, "allow_null": bool}}
    """
    validation_results = []

    for col_name, rules in category_rules.items():
        valid_set = set(rules["valid_values"])
        allow_null = rules.get("allow_null", False)

        # Get actual distinct values
        actual_values = df.select(col_name).distinct().filter(col(col_name).isNotNull())
        actual_set = set([row[col_name] for row in actual_values.collect()])

        # Find invalid values
        invalid_values = actual_set - valid_set

        # Check for nulls if not allowed
        null_count = df.filter(col(col_name).isNull()).count() if not allow_null else 0

        violations = {
            "column": col_name,
            "invalid_values": list(invalid_values),
            "invalid_count": len(invalid_values),
            "null_violations": null_count,
            "status": "PASSED" if (len(invalid_values) == 0 and null_count == 0) else "FAILED"
        }
        validation_results.append(violations)

        # If allowed, add unknown values to a quarantine column for review
        if allow_unknowns and invalid_values:
            df = df.withColumn(
                f"{col_name}_quarantine",
                when(col(col_name).isin(invalid_values), col(col_name)).otherwise(lit(None))
            ).withColumn(
                col_name,
                when(col(col_name).isin(invalid_values), lit("UNKNOWN")).otherwise(col(col_name))
            )

    return df, validation_results

# Usage with dynamic golden set from reference table
def load_golden_set(ref_table_path, column_name):
    """Load valid values from a reference table in Delta Lake"""
    ref_df = spark.read.format("delta").load(ref_table_path)
    return [row["value"] for row in ref_df.select("value").distinct().collect()]

# Dynamic validation against reference data
category_rules = {
    "country_code": {
        "valid_values": load_golden_set("s3://lakehouse/ref/countries", "iso_code"),
        "allow_null": False
    },
    "subscription_tier": {
        "valid_values": ["free", "basic", "premium", "enterprise"],
        "allow_null": True
    },
    "payment_method": {
        "valid_values": ["card", "paypal", "wire", "ach"],
        "allow_null": False
    }
}

df_validated, cat_report = validate_categorical_values(df, category_rules, allow_unknowns=True)

# For detecting new values over time (schema evolution)
def detect_new_categories(df, column, historical_values):
    """Detect new categorical values that appeared since last run"""
    current_values = set([row[column] for row in df.select(column).distinct().collect()])
    new_values = current_values - historical_values

    return new_values, len(new_values)

# 7. Checksum Entire Dataframes for Idempotency


In [8]:
from pyspark.sql.functions import sha2, concat_ws, sum as Fsum, lit
import hashlib

def generate_dataframe_checksum(df, sample_pct=1.0, chunk_size=10000):
    """
    Generate reproducible checksum for entire DataFrame
    sample_pct: percentage of rows to include (for large datasets)
    chunk_size: rows per partition for hashing
    """
    # Sample if needed (for huge datasets)
    if sample_pct < 1.0:
        sample_df = df.sample(withReplacement=False, fraction=sample_pct, seed=42)
    else:
        sample_df = df

    # Convert entire row to string and hash per partition
    def hash_partition(iterator):
        hasher = hashlib.sha256()
        for row in iterator:
            # Create deterministic string representation
            row_str = "|".join([str(v) if v is not None else "NULL" for v in row])
            hasher.update(row_str.encode('utf-8'))
        return [hasher.hexdigest()]

    # Generate per-partition hashes
    partition_hashes = sample_df.rdd.mapPartitions(hash_partition).collect()

    # Combine partition hashes into final checksum
    final_hasher = hashlib.sha256()
    for ph in sorted(partition_hashes):  # Sort for determinism
        final_hasher.update(ph.encode('utf-8'))

    return final_hasher.hexdigest(), len(partition_hashes)

def store_checksum(checksum, pipeline_name, run_date, db_path="checksums.db"):
    """Store checksum in metadata store for comparison"""
    import sqlite3

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS checksums (
            pipeline_name TEXT,
            run_date TEXT,
            checksum TEXT,
            row_count INTEGER,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    cursor.execute(
        "INSERT INTO checksums (pipeline_name, run_date, checksum, row_count) VALUES (?, ?, ?, ?)",
        (pipeline_name, run_date, checksum[0], checksum[1])
    )
    conn.commit()
    conn.close()

def is_duplicate_run(current_checksum, pipeline_name, run_date, db_path="checksums.db"):
    """Check if this data was already processed"""
    import sqlite3

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Check for exact checksum match
    cursor.execute("""
        SELECT COUNT(*) as count FROM checksums
        WHERE pipeline_name = ? AND checksum = ?
    """, (pipeline_name, current_checksum[0]))

    exists = cursor.fetchone()[0] > 0

    # Also check for same date (idempotency)
    cursor.execute("""
        SELECT checksum FROM checksums
        WHERE pipeline_name = ? AND run_date = ?
    """, (pipeline_name, run_date))

    previous_checksum = cursor.fetchone()
    conn.close()

    return exists, previous_checksum

# Usage in incremental pipeline
pipeline_name = "daily_transactions"
today = spark.sql("SELECT current_date()").collect()[0][0]

# Generate checksum (sample 10% for large datasets)
current_checksum = generate_dataframe_checksum(df, sample_pct=0.1)

# Check for duplicates
is_dup, prev_checksum = is_duplicate_run(current_checksum, pipeline_name, str(today))

if is_dup:
    raise ValueError(f"Duplicate data detected. Checksum: {current_checksum[0]}")
else:
    store_checksum(current_checksum, pipeline_name, str(today))
    print(f"New checksum stored: {current_checksum[0]}")

# For streaming idempotency with watermarking
def validate_stream_idempotency(df_microbatch, batch_id, watermark_col="event_time"):
    """Validate microbatch against watermark to prevent reprocessing"""
    # Extract batch time range
    time_range = df_microbatch.select(
        min(col(watermark_col)).alias("min_time"),
        max(col(watermark_col)).alias("max_time")
    ).collect()[0]

    # Check if this time range was already processed
    batch_hash = generate_dataframe_checksum(df_microbatch, sample_pct=0.5)

    # Store in checkpoint location
    checkpoint_path = f"/tmp/stream_checkpoints/batch_{batch_id}_{batch_hash[0]}"

    return batch_hash, time_range

# 8. Validate Cross-Table Referential Integrity


In [9]:
from pyspark.sql.functions import broadcast, count, lit
from pyspark.sql import DataFrame

def validate_referential_integrity(
    df_fact,
    df_dim,
    fk_column,
    pk_column,
    validation_type="strict",
    sample_validation_pct=1.0
):
    """
    Validate referential integrity with performance optimizations
    validation_type: "strict" (all rows), "sample" (percentage), "bloom" (approximate)
    """
    validation_report = {}

    # Total fact records
    total_fact = df_fact.count()
    validation_report["total_fact_records"] = total_fact

    if validation_type == "strict":
        # Use left anti-join to find orphans
        orphan_count = df_fact.join(
            broadcast(df_dim.select(pk_column).distinct()),
            df_fact[fk_column] == df_dim[pk_column],
            "left_anti"
        ).count()

    elif validation_type == "sample":
        # Validate sample for large datasets
        sample_df = df_fact.sample(withReplacement=False, fraction=sample_validation_pct, seed=42)
        orphan_count = int(sample_df.join(
            broadcast(df_dim.select(pk_column).distinct()),
            sample_df[fk_column] == df_dim[pk_column],
            "left_anti"
        ).count() / sample_validation_pct)

    elif validation_type == "bloom":
        # Approximate validation using countDistinct approximation
        distinct_fk = df_fact.select(fk_column).distinct().count()
        distinct_dim = df_dim.select(pk_column).distinct().count()

        # Estimate orphans (approximate)
        orphan_count = max(0, distinct_fk - distinct_dim)

    # Detailed orphan analysis
    if orphan_count > 0:
        orphans = df_fact.join(
            broadcast(df_dim.select(pk_column).distinct()),
            df_fact[fk_column] == df_dim[pk_column],
            "left_anti"
        ).groupBy(fk_column).count().filter(col("count") > 100)  # Focus on high-frequency orphans

        # Store orphan details for investigation
        orphan_details = orphans.collect()

        validation_report.update({
            "orphan_count": orphan_count,
            "orphan_percentage": (orphan_count / total_fact) * 100,
            "top_orphan_keys": orphan_details[:10],
            "status": "FAILED"
        })
    else:
        validation_report.update({
            "orphan_count": 0,
            "orphan_percentage": 0.0,
            "status": "PASSED"
        })

    return validation_report

def validate_multiple_fkeys(df, reference_dict):
    """
    Validate multiple foreign keys against different dimension tables
    reference_dict: {fk_column: {"table": df_dim, "pk": pk_column}}
    """
    all_reports = {}

    for fk, ref_info in reference_dict.items():
        report = validate_referential_integrity(
            df,
            ref_info["table"],
            fk,
            ref_info["pk"],
            validation_type="strict"
        )
        all_reports[fk] = report

    # Overall integrity status
    overall_passed = all(r["status"] == "PASSED" for r in all_reports.values())

    return all_reports, overall_passed

# Usage with multiple dimension tables
ref_tables = {
    "user_id": {"table": spark.table("dim_users"), "pk": "user_id"},
    "product_id": {"table": spark.table("dim_products"), "pk": "product_id"},
    "store_id": {"table": spark.table("dim_stores"), "pk": "store_id"}
}

validation_report, all_valid = validate_multiple_fkeys(df_transactions, ref_tables)

if not all_valid:
    # Log detailed violations and abort
    for fk, report in validation_report.items():
        if report["status"] == "FAILED":
            print(f"Referential integrity violation in {fk}: {report['orphan_count']} orphans")
    raise ValueError("Referential integrity validation failed")

# For slowly changing dimensions (SCD Type 2)
def validate_scd_referential_integrity(df_fact, df_dim_scd, fk_column, sk_column, effective_date_col):
    """
    Validate against SCD Type 2 dimension with validity dates
    """
    # Join on surrogate key and check date ranges
    valid_references = df_fact.join(
        df_dim_scd,
        (df_fact[fk_column] == df_dim_scd[sk_column]) &
        (df_fact[effective_date_col] >= df_dim_scd["valid_from"]) &
        (df_fact[effective_date_col] <= df_dim_scd["valid_to"]),
        "inner"
    )

    valid_count = valid_references.count()
    total_count = df_fact.count()

    return {
        "valid_references": valid_count,
        "total_records": total_count,
        "orphan_percentage": ((total_count - valid_count) / total_count) * 100
    }

# 9. Test for Skewed Partitions Before Writes


In [10]:
from pyspark.sql.functions import spark_partition_id, count as Fcount, col, when
from pyspark.sql.window import Window
import numpy as np

def detect_and_rebalance_skew(df, partition_col, max_skew_factor=10, max_records_per_partition=1_000_000):
    """
    Detect partition skew and rebalance using salting technique
    max_skew_factor: ratio of largest to smallest partition size
    """
    # Analyze current partitioning
    partition_stats = df.withColumn("partition_id", spark_partition_id()) \
        .groupBy("partition_id", partition_col) \
        .agg(Fcount(lit(1)).alias("record_count")) \
        .orderBy(col("record_count").desc())

    # Calculate skew metrics
    stats = partition_stats.select(
        Fcount(lit(1)).alias("num_partitions"),
        Fsum("record_count").alias("total_records"),
        mean(col("record_count")).alias("avg_size"),
        stddev(col("record_count")).alias("stddev_size"),
        min(col("record_count")).alias("min_size"),
        max(col("record_count")).alias("max_size")
    ).collect()[0]

    skew_factor = stats["max_size"] / stats["min_size"] if stats["min_size"] > 0 else float('inf')
    cv = stats["stddev_size"] / stats["avg_size"]  # Coefficient of variation

    # Detect hot keys (partitions exceeding threshold)
    hot_key_threshold = stats["avg_size"] * max_skew_factor
    hot_partitions = partition_stats.filter(col("record_count") > hot_key_threshold)

    skew_report = {
        "total_partitions": stats["num_partitions"],
        "total_records": stats["total_records"],
        "avg_partition_size": stats["avg_size"],
        "skew_factor": skew_factor,
        "coefficient_of_variation": cv,
        "hot_partitions_count": hot_partitions.count(),
        "is_skewed": skew_factor > max_skew_factor or cv > 1.0
    }

    # Rebalancing strategy if skewed
    if skew_report["is_skewed"]:
        # Count distinct values in partition column
        value_counts = df.groupBy(partition_col).count() \
            .orderBy(col("count").desc())

        # Identify high-frequency values to salt
        high_freq_values = value_counts.filter(col("count") > hot_key_threshold) \
            .select(partition_col).rdd.flatMap(lambda x: x).collect()

        # Add salt column for hot keys only
        num_salt_bins = min(10, int(np.ceil(stats["max_size"] / max_records_per_partition)))

        df_salted = df.withColumn(
            "salt",
            when(
                col(partition_col).isin(high_freq_values),
                (rand() * num_salt_bins).cast("int")
            ).otherwise(lit(0))
        )

        # Repartition on salted key
        df_rebalanced = df_salted.repartition(
            stats["num_partitions"] * 2,
            col(partition_col),
            col("salt")
        ).drop("salt")

        return df_rebalanced, skew_report

    return df, skew_report

# Usage with automatic rebalancing
df_balanced, skew_check = detect_and_rebalance_skew(
    df,
    partition_col="customer_id",
    max_skew_factor=5
)

if skew_check["is_skewed"]:
    print(f"Skew detected: factor={skew_check['skew_factor']:.2f}, rebalancing...")
    skew_check["rebalanced"] = True
else:
    print("No significant skew detected")

# For partition size validation against expected range
def validate_partition_sizes(df, partition_col, min_expected_mb=128, max_expected_mb=1024):
    """
    Validate partition sizes in MB against Lakehouse optimization guidelines
    """
    # Write to temporary location and measure
    temp_path = "/tmp/partition_validation"
    df.write.mode("overwrite").parquet(temp_path)

    # Read file sizes from storage
    from pathlib import Path
    sizes_mb = []
    for path in Path(temp_path).rglob("*.parquet"):
        sizes_mb.append(path.stat().st_size / (1024 * 1024))

    # Calculate statistics
    avg_size = np.mean(sizes_mb)
    min_size = np.min(sizes_mb)
    max_size = np.max(sizes_mb)

    too_small = sum(1 for s in sizes_mb if s < min_expected_mb)
    too_large = sum(1 for s in sizes_mb if s > max_expected_mb)

    return {
        "avg_partition_mb": avg_size,
        "min_partition_mb": min_size,
        "max_partition_mb": max_size,
        "too_small_count": too_small,
        "too_large_count": too_large,
        "optimal_range": f"{min_expected_mb}MB - {max_expected_mb}MB"
    }

# 10. Automate Validation with PySpark’s Assert DSL


In [11]:
from pyspark.sql.functions import col, sum as Fsum, when, lit
from typing import Callable, List, Dict, Any
import json

class DataValidationSuite:
    """Custom DSL for PySpark DataFrame validation"""

    def __init__(self, name: str):
        self.name = name
        self.rules: List[Dict[str, Any]] = []
        self.results: List[Dict[str, Any]] = []

    def add_rule(self, description: str, check_func: Callable[[DataFrame], bool], critical: bool = True):
        """Add a validation rule"""
        self.rules.append({
            "description": description,
            "check_func": check_func,
            "critical": critical,
            "rule_id": f"rule_{len(self.rules)}"
        })
        return self

    def run(self, df) -> (bool, List[Dict]):
        """Execute all validation rules"""
        all_passed = True
        self.results = []

        for rule in self.rules:
            try:
                passed = rule["check_func"](df)
                status = "PASSED" if passed else "FAILED"

                if not passed and rule["critical"]:
                    all_passed = False

                self.results.append({
                    "rule_id": rule["rule_id"],
                    "description": rule["description"],
                    "critical": rule["critical"],
                    "status": status
                })
            except Exception as e:
                self.results.append({
                    "rule_id": rule["rule_id"],
                    "description": rule["description"],
                    "critical": rule["critical"],
                    "status": "ERROR",
                    "error_message": str(e)
                })
                if rule["critical"]:
                    all_passed = False

        return all_passed, self.results

    def generate_report(self, output_path: str):
        """Generate JSON validation report"""
        report = {
            "suite_name": self.name,
            "timestamp": spark.sql("SELECT current_timestamp()").collect()[0][0],
            "results": self.results,
            "summary": {
                "total_rules": len(self.results),
                "passed": sum(1 for r in self.results if r["status"] == "PASSED"),
                "failed": sum(1 for r in self.results if r["status"] == "FAILED"),
                "errors": sum(1 for r in self.results if r["status"] == "ERROR")
            }
        }

        with open(output_path, "w") as f:
            json.dump(report, f, default=str, indent=2)

# Build validation suite
validation_suite = DataValidationSuite("daily_sales_validation")

# Add comprehensive rules
validation_suite \
    .add_rule(
        "No negative transaction amounts",
        lambda df: df.filter(col("amount") < 0).count() == 0,
        critical=True
    ) \
    .add_rule(
        "Customer_id exists in dimension table",
        lambda df: df.join(spark.table("dim_customers"), "customer_id", "left_anti").count() == 0,
        critical=True
    ) \
    .add_rule(
        "Transaction count within expected range (10K - 1M)",
        lambda df: 10000 <= df.count() <= 1000000,
        critical=False
    ) \
    .add_rule(
        "Null rate in product_id < 1%",
        lambda df: (df.filter(col("product_id").isNull()).count() / df.count()) < 0.01,
        critical=True
    ) \
    .add_rule(
        "Duplicate transaction_id check",
        lambda df: df.groupBy("transaction_id").count().filter(col("count") > 1).count() == 0,
        critical=True
    ) \
    .add_rule(
        "Revenue sum matches accounting system (within 0.1%)",
        lambda df: abs(df.select(Fsum("amount")).collect()[0][0] - 1_250_000) / 1_250_000 < 0.001,
        critical=False
    )

# Execute validations
all_passed, results = validation_suite.run(df_transactions)

if not all_passed:
    # Log failures and abort
    for result in results:
        if result["status"] in ["FAILED", "ERROR"]:
            print(f"CRITICAL: {result['description']} - {result['status']}")
    raise ValueError("Data validation suite failed")
else:
    # Generate audit report
    validation_suite.generate_report("/lakehouse/validation_reports/daily_validation.json")

# For unit testing PySpark transformations
class PySparkUnitTest:
    """Lightweight unit testing framework for PySpark"""

    @staticmethod
    def assert_schema_equal(df_actual, df_expected):
        assert df_actual.schema == df_expected.schema, f"Schema mismatch:\n{df_actual.schema}\nvs\n{df_expected.schema}"

    @staticmethod
    def assert_row_count_equal(df_actual, expected_count):
        actual = df_actual.count()
        assert actual == expected_count, f"Row count mismatch: expected {expected_count}, got {actual}"

    @staticmethod
    def assert_column_values(df, column, condition, expected_count=0):
        """Assert that number of rows matching condition equals expected_count"""
        actual = df.filter(condition).count()
        assert actual == expected_count, f"Validation failed for {column}: expected {expected_count} rows, got {actual}"

    @staticmethod
    def assert_no_duplicates(df, key_columns):
        """Assert no duplicates exist for given key columns"""
        duplicate_count = df.groupBy(key_columns).count().filter(col("count") > 1).count()
        assert duplicate_count == 0, f"Found {duplicate_count} duplicate groups"

# Example unit test
def test_sales_transformation():
    input_df = spark.createDataFrame([
        (1, 100.0, "A"), (2, 200.0, "B"), (1, 150.0, "A")  # Duplicate id
    ], ["id", "amount", "category"])

    # Run transformation
    result_df = transform_sales(input_df)  # Your transformation function

    # Assertions
    PySparkUnitTest.assert_schema_equal(result_df, expected_schema)
    PySparkUnitTest.assert_no_duplicates(result_df, ["id"])
    PySparkUnitTest.assert_column_values(result_df, "amount", col("amount") < 0, 0)